# Xarray-Spatial Zonal: Sieve filter

Classification outputs often contain salt-and-pepper noise: tiny clumps of a few pixels that don't represent real features. The `sieve` function removes these by replacing connected regions smaller than a threshold with the value of their largest spatial neighbor. It pairs naturally with `natural_breaks()`, `reclassify()`, and `polygonize()`.

### What you'll build

1. [Generate a noisy classified raster](#data)
2. [Basic sieve to remove single-pixel noise](#basic-sieve)
3. [4-connectivity vs 8-connectivity](#connectivity)
4. [Selective sieving with skip_values](#skip-values)
5. [Clean up a natural_breaks classification](#practical)
6. [Compare threshold values](#threshold)

![preview](images/48_sieve_filter.png)

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from xrspatial.sieve import sieve
from xrspatial.classify import natural_breaks

In [ ]:
<a id="data"></a>

## Generate a noisy classified raster

A synthetic raster with three land-cover classes and ~8% salt-and-pepper noise scattered across it.

## Generate a Noisy Classified Raster

We'll create a synthetic classified raster with three land-cover classes and scatter some salt-and-pepper noise across it.

# Teal / blue / orange (colorblind-safe, avoids red/green pairing)
cmap = ListedColormap(['#1abc9c', '#3498db', '#e67e22'])
legend_patches = [
    Patch(facecolor='#1abc9c', label='Class 1'),
    Patch(facecolor='#3498db', label='Class 2'),
    Patch(facecolor='#e67e22', label='Class 3'),
]

fig, ax = plt.subplots(figsize=(8, 5))
raster.plot.imshow(ax=ax, cmap=cmap, vmin=0.5, vmax=3.5, interpolation='nearest', add_colorbar=False)
ax.set_title('Noisy classified raster')
ax.legend(handles=legend_patches, loc='lower right', fontsize=11, framealpha=0.9)
ax.set_axis_off()
plt.tight_layout()

In [ ]:
<a id="basic-sieve"></a>

## Basic sieve: remove single-pixel noise

The simplest use case: set a threshold so isolated pixels are absorbed by their surroundings.

In [ ]:
sieved = sieve(raster, threshold=4)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title in zip(axes, [raster, sieved], ['Before sieve', 'After sieve (threshold=4)']):
    data.plot.imshow(ax=ax, cmap=cmap, vmin=0.5, vmax=3.5, interpolation='nearest', add_colorbar=False)
    ax.set_title(title)
    ax.set_axis_off()
axes[1].legend(handles=legend_patches, loc='lower right', fontsize=11, framealpha=0.9)
plt.tight_layout()

<a id="connectivity"></a>

## Connectivity: 4 vs 8

With 4-connectivity (rook), only pixels sharing an edge count as connected. With 8-connectivity (queen), diagonal neighbors also count. This changes which clumps get flagged as "small."

In [ ]:
sieved_4 = sieve(raster, threshold=6, neighborhood=4)
sieved_8 = sieve(raster, threshold=6, neighborhood=8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, data, title in zip(
    axes,
    [raster, sieved_4, sieved_8],
    ['Original', '4-connectivity (threshold=6)', '8-connectivity (threshold=6)'],
):
    data.plot.imshow(ax=ax, cmap=cmap, vmin=0.5, vmax=3.5, interpolation='nearest', add_colorbar=False)
    ax.set_title(title)
    ax.set_axis_off()
axes[2].legend(handles=legend_patches, loc='lower right', fontsize=11, framealpha=0.9)
plt.tight_layout()

<a id="skip-values"></a>

## Selective sieving with `skip_values`

Sometimes certain classes should never be removed, even if their regions are small. Use `skip_values` to protect specific categories from merging.

In [ ]:
# Protect class 3 from sieving
sieved_skip = sieve(raster, threshold=10, skip_values=[3.0])
sieved_noskip = sieve(raster, threshold=10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, data, title in zip(
    axes,
    [raster, sieved_noskip, sieved_skip],
    ['Original', 'threshold=10 (no skip)', 'threshold=10 (skip class 3)'],
):
    data.plot.imshow(ax=ax, cmap=cmap, vmin=0.5, vmax=3.5, interpolation='nearest', add_colorbar=False)
    ax.set_title(title)
    ax.set_axis_off()
axes[2].legend(handles=legend_patches, loc='lower right', fontsize=11, framealpha=0.9)
plt.tight_layout()

<a id="practical"></a>

## Practical example: clean up a classification

Generate a continuous surface, classify it with `natural_breaks`, and sieve the result to remove small artifacts.

In [ ]:
# Create a smooth surface with some high-frequency variation
y = np.linspace(0, 4 * np.pi, rows)
x = np.linspace(0, 4 * np.pi, cols)
Y, X = np.meshgrid(y, x, indexing='ij')
surface = np.sin(Y) * np.cos(X) + 0.4 * np.random.randn(rows, cols)

surface_da = xr.DataArray(surface, dims=['y', 'x'])
classified = natural_breaks(surface_da, k=5)

# Sieve the classification
cleaned = sieve(classified, threshold=8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
surface_da.plot.imshow(ax=axes[0], cmap='terrain', interpolation='nearest', add_colorbar=False)
axes[0].set_title('Continuous surface')
classified.plot.imshow(ax=axes[1], cmap='tab10', interpolation='nearest', add_colorbar=False)
axes[1].set_title('natural_breaks (k=5)')
cleaned.plot.imshow(ax=axes[2], cmap='tab10', interpolation='nearest', add_colorbar=False)
axes[2].set_title('After sieve (threshold=8)')
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()

<a id="threshold"></a>

## Threshold selection

The right threshold depends on pixel resolution and the minimum feature size you care about. Here is a comparison across values.

In [ ]:
thresholds = [2, 5, 15, 50]
fig, axes = plt.subplots(1, len(thresholds), figsize=(5 * len(thresholds), 5))

for ax, t in zip(axes, thresholds):
    result = sieve(classified, threshold=t)
    result.plot.imshow(ax=ax, cmap='tab10', interpolation='nearest', add_colorbar=False)
    ax.set_title(f'threshold={t}')
    ax.set_axis_off()

plt.suptitle('Effect of sieve threshold on classified raster', y=1.02)
plt.tight_layout()

### References

- [GDAL gdal_sieve.py](https://gdal.org/en/stable/programs/gdal_sieve.html)
- [Connected-component labeling (Wikipedia)](https://en.wikipedia.org/wiki/Connected-component_labeling)
- [xarray-spatial sieve API docs](https://xarray-spatial.readthedocs.io/)

## Threshold Selection

The right threshold depends on pixel resolution and the minimum feature size you care about. Here's a comparison across threshold values.

In [ ]:
thresholds = [2, 5, 15, 50]
fig, axes = plt.subplots(1, len(thresholds), figsize=(5 * len(thresholds), 5))

for ax, t in zip(axes, thresholds):
    result = sieve(classified, threshold=t)
    ax.imshow(result.values, cmap='tab10', interpolation='nearest')
    ax.set_title(f'threshold={t}')

plt.suptitle('Effect of sieve threshold on classified raster', y=1.02)
plt.tight_layout()
plt.show()